# KITTI baseline training (Kaggle GPU)
Thin wrapper: clones the repo and runs the scripts. Needs: GPU on, Internet on, the KITTI dataset attached.
Set `EPOCHS = 3` for the smoke run, then `50` for the baseline.

In [ ]:
# 1. Settings
EPOCHS = 3            # 3 = smoke run, 50 = baseline
RUN_NAME = "smoke" if EPOCHS <= 3 else "baseline"
MODEL = "yolo11n.pt"
IMGSZ = 640
BATCH = 32
KITTI = "/kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset/training"
YOLO_DS = "/kaggle/working/kitti_yolo"
RUNS = "/kaggle/working/runs"

In [ ]:
# 2. Get the code and dependencies
import os
if not os.path.exists("/kaggle/working/kitti-camera-lidar-fusion"):
    !git clone https://github.com/seanpan-1321/kitti-camera-lidar-fusion.git /kaggle/working/kitti-camera-lidar-fusion
%cd /kaggle/working/kitti-camera-lidar-fusion
!git pull
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Split + convert to YOLO format (skip if already done)
if not os.path.exists(f"{YOLO_DS}/labels/val"):
    !python scripts/make_split.py --kitti_root {KITTI} --out /kaggle/working/splits
    !python scripts/convert_kitti_to_yolo.py --kitti_root {KITTI} --splits /kaggle/working/splits --out {YOLO_DS}
!ls {YOLO_DS}/images/train | wc -l
!ls {YOLO_DS}/images/val | wc -l

In [ ]:
# 4. Train. `timeout 3h` kills a stalled run instead of letting it burn GPU quota;
# last.pt/best.pt are saved every epoch, so the cells below still work after a timeout.
!timeout 3h yolo detect train data=configs/kitti.yaml model={MODEL} imgsz={IMGSZ} epochs={EPOCHS} batch={BATCH} workers=2 seed=42 project={RUNS} name={RUN_NAME} exist_ok=True

In [ ]:
# 5. Per-class validation metrics (copy these numbers into results/log.md)
import glob
best = glob.glob(f"/kaggle/working/**/{RUN_NAME}/weights/best.pt", recursive=True)[0]
print(best)
!yolo detect val data=configs/kitti.yaml model={best} imgsz={IMGSZ} split=val

In [ ]:
# 6. Bundle outputs to download: best.pt + results.csv
import shutil
run_dir = os.path.dirname(os.path.dirname(best))
os.makedirs(f"/kaggle/working/export_{RUN_NAME}", exist_ok=True)
shutil.copy(best, f"/kaggle/working/export_{RUN_NAME}/best.pt")
shutil.copy(f"{run_dir}/results.csv", f"/kaggle/working/export_{RUN_NAME}/results.csv")
print(os.listdir(f"/kaggle/working/export_{RUN_NAME}"))